# 08 — Analisis cualitativo

**Librerias usadas:** pandas, matplotlib, json (estandar)

In [ ]:
import sys, os, json
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'solineras'))
os.chdir(ROOT)
DATA = ROOT / 'data'

CODIGOS = {
    'falta_repuestos': ['repuesto', 'pieza', 'escasez'],
    'clima_adverso': ['huracan', 'lluvia', 'nublado', 'tormenta'],
    'almacenamiento': ['bateria', 'almacenamiento', 'no alcanza'],
    'mantenimiento': ['mantenimiento', 'limpieza', 'revision', 'averia'],
    'costo': ['caro', 'costo', 'precio'],
    'conocimiento_tecnico': ['no se', 'no entiendo', 'capacitacion'],
    'red_electrica': ['apagon', 'red', 'conexion'],
    'satisfaccion_positiva': ['bien', 'excelente', 'bueno', 'contento'],
}

def codificar(texto):
    if not isinstance(texto, str):
        return []
    t = texto.lower()
    return [c for c, palabras in CODIGOS.items() if any(p in t for p in palabras)]

## 1. Codificar

In [ ]:
def leer_jsonl(p):
    if not p.exists():
        return []
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

registros = []
for tipo, arch in [('solinera', 'entrevistas_solineras.jsonl'),
                   ('centro', 'entrevistas_centros.jsonl'),
                   ('residente', 'entrevistas_residentes.jsonl')]:
    for doc in leer_jsonl(DATA / 'raw' / arch):
        sid = doc.get('id_solinera') or doc.get('id_centro') or doc.get('id_residente')
        for seccion, contenido in doc.items():
            if isinstance(contenido, dict):
                for campo, valor in contenido.items():
                    if isinstance(valor, str) and len(valor) > 3:
                        for c in codificar(valor):
                            registros.append({'id_entidad': sid, 'tipo_entidad': tipo,
                                              'seccion': seccion, 'campo': campo,
                                              'texto': valor, 'codigo': c})
cuali = pd.DataFrame(registros)
print('Citas codificadas:', len(cuali))

## 2. Frecuencias

In [ ]:
if not cuali.empty:
    freq = cuali['codigo'].value_counts()
    display(freq)
    plt.figure(figsize=(10, 5))
    freq.plot(kind='barh', color='steelblue')
    plt.title('Frecuencia de codigos tematicos')
    plt.xlabel('Citas')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('Sin entrevistas. Anade datos en data/raw/')

## 3. Por tipo

In [ ]:
if not cuali.empty:
    display(cuali.groupby(['tipo_entidad','codigo']).size().unstack(fill_value=0))